In [1]:
import os
import sys

# Add the specific directory containing your local module
sys.path.insert(0, os.path.abspath(os.path.join("./TimeSformer")))

In [2]:
import torch
from timesformer.models.vit import TimeSformer

model = TimeSformer(img_size=224, num_classes=400, num_frames=8, attention_type='divided_space_time',  pretrained_model='./pretrained/TimeSformer_divST_8x32_224_K400.pyth')

dummy_video = torch.randn(2, 3, 8, 224, 224) # (batch x channels x frames x height x width)

pred = model(dummy_video,) # (2, 400)

c:\Users\darkn\miniconda3\envs\timesformer\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
pred.shape

torch.Size([2, 400])

In [4]:
from ProfVLM.model import ProfVLM

prof_vlm = ProfVLM(pretrained_model='./pretrained/TimeSformer_divST_8x32_224_K400.pyth')
prof_vlm(dummy_video)

tensor([[-0.2221, -0.2732,  0.6756,  ...,  0.1924, -0.1900,  0.2851],
        [ 0.5344,  0.2713,  1.1845,  ...,  0.2842, -0.6394,  1.4078]],
       grad_fn=<AddmmBackward0>)

### SkillFormer Ego

Overall test accuracy: 0.4454  

Accuracy by scenario:  
  Basketball: 0.4545 (55 samples)  
  Bouldering: 0.2658 (79 samples)  
  Cooking: 0.4167 (12 samples)  
  Dance: 0.5510 (49 samples)  
  Music: 0.7857 (28 samples)  
  Soccer: 0.3333 (6 samples)  

Overall predicted class distribution:  
  Class 0: 20 samples (8.73%)  
  Class 1: 86 samples (37.55%)  
  Class 2: 54 samples (23.58%)  
  Class 3: 69 samples (30.13%)  

### Skillformer Exo

Overall test accuracy: 0.4454  

Accuracy by scenario:  
  Basketball: 0.6000 (55 samples)  
  Bouldering: 0.2532 (79 samples)  
  Cooking: 0.5000 (12 samples)  
  Dance: 0.4490 (49 samples)  
  Music: 0.6786 (28 samples)  
  Soccer: 0.3333 (6 samples)  

Overall predicted class distribution:  
  Class 0: 35 samples (15.28%)  
  Class 1: 91 samples (39.74%)  
  Class 2: 41 samples (17.90%)  
  Class 3: 62 samples (27.07%)  

In [ ]:
import json

with open("data/annotations/proficiency_demonstrator_train.json", "r") as file:
    proficiency_metadata = json.load(file)["annotations"]

In [ ]:
import re
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

LEVELS = {
    "Novice": 1, 
    "Early Expert": 2, 
    "Intermediate Expert": 3, 
    "Late Expert": 4,
}

# Expert commentary lives at <COMMENTARY_ROOT>/<take_name>/**/data.json, with the
# transcribed text in a transcriptions.json next to each data.json.
# If unsure where it is: find data -name transcriptions.json | head
COMMENTARY_ROOT = Path("data/annotations/expert_commentary")

# Experts sometimes say the label outright ("this is a novice"); mask it so the
# model can't just read the answer off the commentary.
RATING_TERMS = re.compile(
    r"\b(?:novice|early\s+expert|intermediate\s+expert|late\s+expert|"
    r"proficien\w*|rating|score|good\s+execution|tip\s+for\s+improvement|beginner)\b",
    flags=re.IGNORECASE,
)

def load_commentary(take_name):
    """Return the take's expert commentary as timestamped lines, or None if there is none."""
    segments = []
    for data_path in (COMMENTARY_ROOT / take_name).rglob("data.json"):
        transcript_path = data_path.parent / "transcriptions.json"
        if not transcript_path.exists():
            continue
        data = json.loads(data_path.read_text(encoding="utf-8"))
        transcripts = json.loads(transcript_path.read_text(encoding="utf-8"))
        for annotation in data.get("annotations", []):
            text = transcripts.get(annotation.get("recording_path"), {}).get("text", "").strip()
            if text:
                segments.append((annotation.get("video_time") or 0.0, RATING_TERMS.sub("[MASK]", text)))
    if not segments:
        return None
    return "\n".join(f"[{t:.1f}s] {text}" for t, text in sorted(segments))

class PromptDataset(Dataset):
    def __init__(self, proficiency_metadata, video_type='ego', require_commentary=True):
        if not COMMENTARY_ROOT.exists():
            raise FileNotFoundError(f"{COMMENTARY_ROOT} not found; point COMMENTARY_ROOT at the expert_commentary folder")
        self.paths, self.commentary, self.targets = [], [], []
        for m in proficiency_metadata:
            video_path = m['video_paths'][video_type].split('/')
            video_path[-1:-1] = ['downscaled', '448']
            video_path = f"data/{'/'.join(video_path)}"
            if not os.path.exists(video_path):
                continue
            commentary = load_commentary(m['take_name'])
            if commentary is None and require_commentary:
                continue
            self.paths.append(video_path)
            self.commentary.append(commentary or "(no commentary available)")
            self.targets.append(LEVELS[m['proficiency_score']])

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        messages = [
            {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
            {"role": "user", "content": [
                {"type": "video", "video": self.paths[idx]},
                {"type": "text", "text": f"Expert commentary:\n{self.commentary[idx]}"},
            ]},
            {"role": "assistant", "content": [{"type": "text", "text": f"Score: {self.targets[idx]}"}]},
        ]
        return {"messages": messages, "score": self.targets[idx]}

system_prompt = """You assess the proficiency of a performer from a video and an expert's commentary on that video.

Base your judgment on what is observable in the video. Use the commentary as supporting evidence about technique, not as a substitute for watching.

Proficiency scale:
1 - Novice
2 - Early Expert
3 - Intermediate Expert
4 - Late Expert

Respond in exactly this format and nothing else:
Score: <single digit 1-4>
"""

with open("data/annotations/proficiency_demonstrator_val.json", "r") as file:
    proficiency_metadata_val = json.load(file)["annotations"]

train_dataset = PromptDataset(proficiency_metadata)
val_dataset = PromptDataset(proficiency_metadata_val)
print(f"train: {len(train_dataset)} takes, val: {len(val_dataset)} takes")

In [ ]:
import torch
from transformers import AutoModelForMultimodalLM, AutoProcessor, Trainer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "Qwen/Qwen3.5-9B"
NUM_FRAMES = 16  # frames sampled per video; more frames = more visual tokens

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForMultimodalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
processor = AutoProcessor.from_pretrained(model_id)
processor.tokenizer.padding_side = "right"

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

# Only the assistant's answer contributes to the loss: mask everything up to and
# including the last "<|im_start|>assistant\n" header, plus padding.
ASSISTANT_HEADER = processor.tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)

def collate_fn(examples):
    batch = processor.apply_chat_template(
        [ex["messages"] for ex in examples],
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
        padding=True,
        num_frames=NUM_FRAMES,
    )
    labels = batch["input_ids"].clone()
    labels[batch["attention_mask"] == 0] = -100
    n = len(ASSISTANT_HEADER)
    for i, ids in enumerate(batch["input_ids"].tolist()):
        starts = [j for j in range(len(ids) - n + 1) if ids[j:j + n] == ASSISTANT_HEADER]
        labels[i, :starts[-1] + n] = -100
    batch["labels"] = labels
    return batch

training_args = TrainingArguments(
    output_dir="./qwen35-9b-finetuned",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    bf16=True,  # Recommended over fp16 for native Qwen3.5 stability
    logging_steps=10,
    max_steps=100,  # Adjust to your dataset size/epochs
    save_strategy="steps",
    save_steps=50,
    eval_strategy="steps",
    eval_steps=50,
    per_device_eval_batch_size=2,
    optim="paged_adamw_8bit",
    gradient_checkpointing=True,
    remove_unused_columns=False,  # keep "messages" for collate_fn
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=collate_fn,
)

trainer.train()

trainer.model.save_pretrained("./qwen35-9b-adapter")



# Inference on a held-out take: drop the assistant turn and let the model answer
example = val_dataset[0]
inputs = processor.apply_chat_template(
	example["messages"][:-1],
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt",
	num_frames=NUM_FRAMES,
).to(model.device)

outputs = model.generate(**inputs, max_new_tokens=16)
print("true:", example["score"])
print(processor.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))